In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd

df_pairs = pd.read_csv('/content/drive/MyDrive/NLP-Job-Screener/matching_pairs.csv')
df_resume = pd.read_csv('/content/drive/MyDrive/NLP-Job-Screener/resume_clean.csv')
df_jobs = pd.read_parquet('/content/drive/MyDrive/NLP-Job-Screener/jobs_clean.parquet')


In [5]:
print(df_pairs.shape)
print(df_pairs.head(3))

(29796, 4)
                                         resume_text  \
0  hr administrator marketing associate hr admini...   
1  hr administrator marketing associate hr admini...   
2  hr administrator marketing associate hr admini...   

                                           job_title  \
0                         Human Resources Generalist   
1  TWIC On-Call Crew Transportation Driver - Corp...   
2  Sales Associate - The Cosmetics Company Store ...   

                                     job_description  label  
0  job description with supervision the hr genera...      1  
1  danner s inc a leading maritime transportation...      1  
2  position summary as one of our highly skilled ...      1  


In [6]:
!pip install sentence-transformers

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2', device="cuda")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
df_pairs.head(3)

,resume_text,job_title,job_description,label
0,hr administrator marketing associate hr admini...,Human Resources Generalist,job description with supervision the hr genera...,1
1,hr administrator marketing associate hr admini...,TWIC On-Call Crew Transportation Driver - Corp...,danner s inc a leading maritime transportation...,1
2,hr administrator marketing associate hr admini...,Sales Associate - The Cosmetics Company Store ...,position summary as one of our highly skilled ...,1


In [9]:
df_pairs['job_title_description'] = df_pairs['job_title'].str.lower() + " " + df_pairs['job_description']


resume_embeddings = model.encode(df_pairs['resume_text'].to_list(), convert_to_tensor=True)


job_embeddings = model.encode(df_pairs['job_title_description'].to_list(), convert_to_tensor=True)

In [10]:
print(resume_embeddings.shape)
print(job_embeddings.shape)

torch.Size([29796, 384])
torch.Size([29796, 384])


In [11]:
from sentence_transformers import util
cosine_scores = util.cos_sim(resume_embeddings, job_embeddings).diagonal()
print(cosine_scores.shape)
print(cosine_scores[:5])

torch.Size([29796])
tensor([0.4459, 0.3317, 0.4119, 0.2216, 0.4358], device='cuda:0')


In [12]:
df_pairs['cosine_score'] = cosine_scores.cpu().numpy()

In [13]:
df_pairs.head(10)

,resume_text,job_title,job_description,label,job_title_description,cosine_score
0,hr administrator marketing associate hr admini...,Human Resources Generalist,job description with supervision the hr genera...,1,human resources generalist job description wit...,0.445917
1,hr administrator marketing associate hr admini...,TWIC On-Call Crew Transportation Driver - Corp...,danner s inc a leading maritime transportation...,1,twic on-call crew transportation driver - corp...,0.331656
2,hr administrator marketing associate hr admini...,Sales Associate - The Cosmetics Company Store ...,position summary as one of our highly skilled ...,1,sales associate - the cosmetics company store ...,0.411923
3,hr administrator marketing associate hr admini...,Quality Supervisor,location sumter sc about skf skf has been maki...,0,quality supervisor location sumter sc about sk...,0.221553
4,hr administrator marketing associate hr admini...,Corporate Director of Reimbursement (Exempt),facility cape fear valley medical center locat...,0,corporate director of reimbursement (exempt) f...,0.435810
5,hr administrator marketing associate hr admini...,Treatment Plan Coordinator,treatment plan coordinator tpc treatment plann...,0,treatment plan coordinator treatment plan coor...,0.425430
6,hr administrator marketing associate hr admini...,Dialysis Registered Nurse,how you ll change lives as a registered nurse ...,0,dialysis registered nurse how you ll change li...,0.274018
7,hr administrator marketing associate hr admini...,Senior Compliance Officer - Alternatives,employer dws grouptitle senior afc compliance ...,0,senior compliance officer - alternatives emplo...,0.344020
8,hr administrator marketing associate hr admini...,Echo Sonographer - Cardiac,our client is looking to add an echo sonograph...,0,echo sonographer - cardiac our client is looki...,0.390519
9,hr administrator marketing associate hr admini...,Postdoctoral Researcher,the liver cancer program at icahn school of me...,0,postdoctoral researcher the liver cancer progr...,0.106998


In [14]:
def skill_overlap(resume_text, job_title_description):
  resume_text = set(resume_text.split())
  job_title_description = set(job_title_description.split())

  common = job_title_description.intersection(resume_text)
  ratio = len(common) / len(job_title_description) if job_title_description else 0
  return ratio

df_pairs['skill_overlap'] = df_pairs.apply(
    lambda row: skill_overlap(row['resume_text'], row['job_title_description']), axis=1
)

df_pairs.head(3)

,resume_text,job_title,job_description,label,job_title_description,cosine_score,skill_overlap
0,hr administrator marketing associate hr admini...,Human Resources Generalist,job description with supervision the hr genera...,1,human resources generalist job description wit...,0.445917,0.201946
1,hr administrator marketing associate hr admini...,TWIC On-Call Crew Transportation Driver - Corp...,danner s inc a leading maritime transportation...,1,twic on-call crew transportation driver - corp...,0.331656,0.259615
2,hr administrator marketing associate hr admini...,Sales Associate - The Cosmetics Company Store ...,position summary as one of our highly skilled ...,1,sales associate - the cosmetics company store ...,0.411923,0.210884


In [15]:
print(df_pairs['skill_overlap'].describe())

count    29796.000000
mean         0.202273
std          0.066187
min          0.000000
25%          0.157530
50%          0.195122
75%          0.239451
max          0.588235
Name: skill_overlap, dtype: float64


In [16]:
def text_length_ratio(resume_text, job_title_description):
  if len(resume_text) > len(job_title_description):
    res = len(job_title_description) / len(resume_text)
  elif len(job_title_description) > len(resume_text):
    res = len(resume_text) / len(job_title_description)
  else:
    res = 1
  return res

In [17]:
df_pairs['text_lenght_ratio'] = df_pairs.apply(
    lambda row: text_length_ratio(row['resume_text'], row['job_title_description']), axis=1
)
df_pairs.head()

,resume_text,job_title,job_description,label,job_title_description,cosine_score,skill_overlap,text_lenght_ratio
0,hr administrator marketing associate hr admini...,Human Resources Generalist,job description with supervision the hr genera...,1,human resources generalist job description wit...,0.445917,0.201946,0.919221
1,hr administrator marketing associate hr admini...,TWIC On-Call Crew Transportation Driver - Corp...,danner s inc a leading maritime transportation...,1,twic on-call crew transportation driver - corp...,0.331656,0.259615,0.184606
2,hr administrator marketing associate hr admini...,Sales Associate - The Cosmetics Company Store ...,position summary as one of our highly skilled ...,1,sales associate - the cosmetics company store ...,0.411923,0.210884,0.749537
3,hr administrator marketing associate hr admini...,Quality Supervisor,location sumter sc about skf skf has been maki...,0,quality supervisor location sumter sc about sk...,0.221553,0.188679,0.928885
4,hr administrator marketing associate hr admini...,Corporate Director of Reimbursement (Exempt),facility cape fear valley medical center locat...,0,corporate director of reimbursement (exempt) f...,0.435810,0.278788,0.396995


In [18]:
X = df_pairs[['cosine_score', 'skill_overlap', 'text_lenght_ratio']]
y = df_pairs['label']

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=42,
    test_size=0.3
)

In [20]:
from sklearn.utils import class_weight
import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=100,
    scale_pos_weight=3,
    random_state=42
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

In [21]:
from sklearn.metrics import classification_report
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.74      0.78      6733
           1       0.39      0.51      0.44      2206

    accuracy                           0.68      8939
   macro avg       0.61      0.63      0.61      8939
weighted avg       0.72      0.68      0.70      8939



In [22]:
import numpy as np

resume_np = resume_embeddings.cpu().numpy()
job_np = job_embeddings.cpu().numpy()

diff = np.abs(resume_np - job_np)

X_emb = np.hstack([
    df_pairs[['cosine_score', 'skill_overlap', 'text_lenght_ratio']].values,
    diff
])

X_train, X_test, y_train, y_test = train_test_split(
    X_emb,
    y,
    random_state=42,
    test_size=0.3
)

from sklearn.utils import class_weight
import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=100,
    scale_pos_weight=3,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.87      0.85      6733
           1       0.52      0.44      0.48      2206

    accuracy                           0.76      8939
   macro avg       0.67      0.65      0.66      8939
weighted avg       0.75      0.76      0.75      8939



In [23]:
import joblib
joblib.dump(model, 'matching_model.joblib')

['matching_model.joblib']